#***Integración y Visualización Dinámica de Datos Estadísticos Oficiales***

Este proyecto tiene como objetivo fundamental superar las barreras de interoperabilidad en la información estadística oficial. Se centra en la normalización de datasets de fuentes oficiales diversas, creando un marco que garantiza la disponibilidad y la coherencia de los datos para permitir análisis robustos e interconectados, independientemente de su origen primario.

---

***Exploración de Datos Interconectados***

Integrado en este cuaderno, se presenta un entorno de interactivo que ofrece las siguientes opciones:

* Filtro Temporal: Aplicación de filtros de fechas intuitivos para observar la evolución de las series de datos a lo largo del tiempo.

* Selección de Variables: Permite al usuario identificar y unificar variables de interés procedentes de múltiples datasets estandarizados, con la capacidad de poder mostrar hasta 6 series pudiendo seleccionar si los datos representan valores o porcentajes para poder generar un eje Y secundario.

* Visualización Gráfica: Generación de gráficos dinámicos que reflejan la evolución temporal de los datos seleccionados.

* Exportación Customizada: Posibilidad de descargar tanto la gráfica generada como el dataset custom (conjunto de datos personalizado) que contiene únicamente la información filtrada y unificada.

El resultado es una herramienta facilitadora para el análisis de tendencias históricas, haciendo accesible la información compleja y heterogénea a través de una interfaz accesible.

---

***Autor: Martín Nicolás Serafini***

Linkedin: ***https://www.linkedin.com/in/martin-nicolas-serafini-05224923b/***

Github: ***https://github.com/MartinSerafini***


In [5]:
# ---------------------------
# IMPORTACION DE LIBRERIAS
# ---------------------------

import os, glob, datetime
from io import BytesIO, StringIO
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from google.colab import files
import ipywidgets as widgets
from IPython.display import display, clear_output

# ---------------------------
# Monto drive
# ---------------------------
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [6]:
# ------------------------------------------------------------------------------------
# CONFIGURACION INICIAL - IMPORTANTE - PEGAR LA RUTA PARA PERMITIR LA LECTURA DE DATOS
# ------------------------------------------------------------------------------------

DATA_DIR = "/.../data_clean" # Copiar la ruta completa del directorio /data_clean tal cual figura en su Google Drive
DATE_COL = "periodo"

In [7]:
# ---------------------------
# LECTURA AUTOMÁTICA DE CSV
# ---------------------------

csv_paths = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))
csv_files = [os.path.basename(p) for p in csv_paths]

datasets = {}
for p, name in zip(csv_paths, csv_files):
    df = pd.read_csv(p)
    df = df.copy()
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
    df = df.dropna(subset=[DATE_COL]).sort_values(DATE_COL).reset_index(drop=True)
    datasets[name] = df

global_min_date = min(df[DATE_COL].min() for df in datasets.values())
global_max_date = max(df[DATE_COL].max() for df in datasets.values())

# ---------------------------
# ADICIONALES GRAFICOS
# ---------------------------
def choose_xticks(dates_index, max_ticks=12):
    total = len(dates_index)
    if total <= max_ticks:
        return dates_index
    idx = np.round(np.linspace(0, total-1, max_ticks)).astype(int)
    return dates_index[idx]

COLOR_OPTIONS = ['blue','red','green','orange','purple','black','brown','cyan','magenta']
LINESTYLES = {"Línea continua": "-", "Punteada": "--", "Puntos": ":", "Guion-puntos": "-."}

# ---------------------------
# WIDGETS PRINCIPALES
# ---------------------------
fecha_desde = widgets.DatePicker(
    description="Desde",
    value=global_min_date,
    min=global_min_date,
    max=global_max_date
)

fecha_hasta = widgets.DatePicker(
    description="Hasta",
    value=global_max_date,
    min=global_min_date,
    max=global_max_date
)

n_series_sel = widgets.Dropdown(
    options=[1,2,3,4,5,6],
    value=3,
    description="Series"
)

series_container = widgets.VBox(layout=widgets.Layout(width="1200px"))
grafico_out = widgets.Output()
download_buttons_out = widgets.Output()

# ---------------------------
# TITULO & SUBTITULO
# ---------------------------
titulo_cb = widgets.Checkbox(description="Agrega Título?", value=False, layout=widgets.Layout(width="300px"))
titulo_text = widgets.Text(description="Título:", placeholder="Máx 80 caracteres",
                           layout=widgets.Layout(width="800px"), max_length=80)

subtitulo_cb = widgets.Checkbox(description="Agrega Subtítulo?", value=False, layout=widgets.Layout(width="300px"))
subtitulo_text = widgets.Text(description="Subtítulo:", placeholder="Máx 120 caracteres",
                              layout=widgets.Layout(width="800px"), max_length=120)

grid_v_cb = widgets.Checkbox(description="Grid vertical", value=False, layout=widgets.Layout(width="300px"))

titulo_text.layout.display = "none"
subtitulo_text.layout.display = "none"

def _toggle_titulo(change):
    titulo_text.layout.display = "block" if change["new"] else "none"

def _toggle_subtitulo(change):
    subtitulo_text.layout.display = "block" if change["new"] else "none"

titulo_cb.observe(_toggle_titulo, names="value")
subtitulo_cb.observe(_toggle_subtitulo, names="value")

# ---------------------------
# CREACIÓN DE PANEL POR SERIE
# ---------------------------
def create_series_widgets(index):

    dataset_dd = widgets.Dropdown(
        options=csv_files,
        layout=widgets.Layout(width="250px")
    )

    campo_dd = widgets.Dropdown(
        options=[],
        layout=widgets.Layout(width="300px")
    )

    estilo_dd = widgets.Dropdown(
        options=list(LINESTYLES.keys()),
        layout=widgets.Layout(width="120px")
    )

    color_dd = widgets.Dropdown(
        options=COLOR_OPTIONS,
        value=COLOR_OPTIONS[index % len(COLOR_OPTIONS)],
        layout=widgets.Layout(width="80px")
    )

    pct_cb = widgets.Checkbox(
        value=False,
        indent=False,
        layout=widgets.Layout(width="30px", margin="0 0 0 5px")
    )

    # ----- actualizar CAMPOS según dataset seleccionado -----
    def update_fields(change):
        dname = change["new"]
        if dname not in datasets:
            campo_dd.options = []
            return
        cols = sorted([c for c in datasets[dname].columns if c != DATE_COL])
        campo_dd.options = cols
        if cols:
            campo_dd.value = cols[0]

    dataset_dd.observe(update_fields, names="value")
    update_fields({"new": dataset_dd.value})

    # ----- fila de configuración -----
    fila = widgets.HBox(
        [
            widgets.Label(f"{index+1}", layout=widgets.Layout(width="25px")),
            dataset_dd,
            campo_dd,
            estilo_dd,
            color_dd,
            pct_cb
        ],
        layout=widgets.Layout(align_items="center")
    )

    return {
        "box": fila,
        "dataset": dataset_dd,
        "campo": campo_dd,
        "estilo": estilo_dd,
        "color": color_dd,
        "pct": pct_cb
    }

# ---------------------------
# GRAFICA
# ---------------------------
def on_graficar_clicked(b):
    grafico_out.clear_output()
    download_buttons_out.clear_output()
    plt.close('all')

    if fecha_desde.value is None or fecha_hasta.value is None:
        with grafico_out: print("Seleccioná fechas.")
        return

    fd = pd.Timestamp(fecha_desde.value)
    fh = pd.Timestamp(fecha_hasta.value)

    if fd > fh:
        with grafico_out: print("Rango inválido.")
        return

    series_widgets = getattr(series_container, "series_widgets", [])
    if not series_widgets:
        with grafico_out: print("No hay series definidas.")
        return

    autor = "Martin Nicolas Serafini"
    master_idx = build_master_index(fd, fh)

    collected = pd.DataFrame(index=master_idx)
    collected.index.name = DATE_COL

    has_pct = False
    has_abs = False
    series_info = []

    for w in series_widgets:
        dname = w["dataset"].value
        campo = w["campo"].value
        if not dname or not campo:
            continue

        df = datasets[dname].copy()
        df[DATE_COL] = df[DATE_COL].dt.to_period("M").dt.to_timestamp()
        dff = df.set_index(DATE_COL)

        y = dff.reindex(master_idx)[campo] if campo in dff.columns else pd.Series(index=master_idx, dtype=float)
        series_name = f"{dname.replace('.csv','')} - {campo}"
        collected[series_name] = y.values

        is_pct = bool(w["pct"].value)
        if is_pct: has_pct = True
        else: has_abs = True

        series_info.append({
            "name": series_name,
            "x": master_idx,
            "y": y.values,
            "is_pct": is_pct,
            "estilo": LINESTYLES[w["estilo"].value],
            "color": w["color"].value
        })

    if not series_info:
        with grafico_out: print("Nada para graficar.")
        return

    with grafico_out:
        fig, ax = plt.subplots(figsize=(14,6))
        ax2 = None
        left_lines = []
        right_lines = []
        left_labels = []
        right_labels = []

        if grid_v_cb.value:
            ax.grid(axis="x", alpha=0.7)

        for s in series_info:
            if has_pct and has_abs and s["is_pct"]:
                if ax2 is None:
                    ax2 = ax.twinx()
                ln, = ax2.plot(s["x"], s["y"], s["estilo"], color=s["color"], label=s["name"])
                right_lines.append(ln); right_labels.append(s["name"])
            else:
                ln, = ax.plot(s["x"], s["y"], s["estilo"], color=s["color"], label=s["name"])
                left_lines.append(ln); left_labels.append(s["name"])

        ticks = choose_xticks(master_idx)
        ax.set_xticks(ticks)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%Y"))
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

        if titulo_cb.value and titulo_text.value.strip():
            fig.suptitle(titulo_text.value.strip().upper(), fontsize=18, y=1.01)

        if subtitulo_cb.value and subtitulo_text.value.strip():
            fig.text(0.5, 0.95, subtitulo_text.value.strip(), ha='center', fontsize=12)

        if has_pct and not has_abs:
            ax.set_ylabel("Porcentaje (%)")
        elif has_abs and not has_pct:
            ax.set_ylabel("Valor")
        else:
            ax.set_ylabel("Valor")
            if ax2: ax2.set_ylabel("Porcentaje (%)")

        all_lines = left_lines + right_lines
        all_labels = left_labels + right_labels
        fig.legend(all_lines, all_labels, loc='lower center',
                   ncol=min(3,len(all_lines)), bbox_to_anchor=(0.5, -0.03))

        fig.text(1, -0.065,
                 f"Datos: INDEC. Gráfico y Elaboración: {autor}\nEste Reporte se distribuye bajo Licencia Creative Commons CC BY-SA 4.0.",
                 ha='right', fontsize=8, color='gray')

        plt.tight_layout(rect=[0, 0.06, 1, 0.95])
        plt.show()
        plt.close(fig)

    with download_buttons_out:
        download_buttons_out.clear_output()

        def download_png_fn(_):
            png_path = "grafico_generado.png"
            fig.savefig(png_path, dpi=150, bbox_inches='tight')
            files.download(png_path)

        def download_csv_fn(_):
            fecha_txt = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
            header = (
                f"Dataset generado en base a datos oficiales (INDEC)\n"
                f"Autor: {autor}\n"
                f"Fecha: {fecha_txt}\n"
                f'Este Reporte se distribuye bajo Licencia Creative Commons CC BY-SA 4.0.\n\n'
            )
            buf = StringIO()
            buf.write(header)
            df_out = collected.reset_index()
            df_out.to_csv(buf, index=False)

            nombre_csv = f'dataset_{fecha_txt}.csv'
            csv_path = nombre_csv
            with open(csv_path, "w", encoding="utf-8") as f:
                f.write(buf.getvalue())

            files.download(csv_path)

        btn_png = widgets.Button(description="Descargar PNG", button_style="info",
                                 layout=widgets.Layout(width="160px"))
        btn_csv = widgets.Button(description="Descargar CSV", button_style="warning",
                                 layout=widgets.Layout(width="160px"))

        btn_png.on_click(download_png_fn)
        btn_csv.on_click(download_csv_fn)

        display(widgets.HBox([btn_png, btn_csv]))

# ------------------------------------------
# ACTUALIZA NUMERO DE SERIES AUTOMÁTICAMENTE
# ------------------------------------------
def update_series(_):
    grafico_out.clear_output()
    download_buttons_out.clear_output()

    n = n_series_sel.value
    series_widgets = []

    header = widgets.HBox([
        widgets.Label("N°", layout=widgets.Layout(width="30px")),
        widgets.Label("Dataset", layout=widgets.Layout(width="250px")),
        widgets.Label("Campo", layout=widgets.Layout(width="300px")),
        widgets.Label("Estilo", layout=widgets.Layout(width="120px")),
        widgets.Label("Color", layout=widgets.Layout(width="80px")),
        widgets.Label("%", layout=widgets.Layout(width="30px"))
    ], layout=widgets.Layout(width="1200px"))   # *** FIX ***

    for i in range(n):
        series_widgets.append(create_series_widgets(i))

    btn_graficar = widgets.Button(description="Graficar", button_style="success")
    btn_graficar.on_click(on_graficar_clicked)

    series_container.children = (
        [header] +
        [w["box"] for w in series_widgets] +
        [
            widgets.HBox([titulo_cb, titulo_text]),
            widgets.HBox([subtitulo_cb, subtitulo_text]),
            widgets.HBox([grid_v_cb]),
            widgets.Label("", layout=widgets.Layout(height="6px")),
            btn_graficar
        ]
    )

    series_container.series_widgets = series_widgets

n_series_sel.observe(update_series, names="value")
update_series(None)

# ---------------------------
# CONSTRUYE INDICE MENSUAL
# ---------------------------
def build_master_index(fd, fh):
    start = pd.Timestamp(fd).to_period('M').to_timestamp()
    end = pd.Timestamp(fh).to_period('M').to_timestamp()
    return pd.date_range(start=start, end=end, freq='MS')



# ---------------------------
# MOSTRAR INTERFAZ
# ---------------------------
display(
    widgets.VBox([
        widgets.HBox([fecha_desde, fecha_hasta]),
        n_series_sel,
        series_container,
        grafico_out,
        download_buttons_out
    ])
)



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>